# Inventory policies – CA_1, FOODS_3

Goal: turn the Phase 1 forecasts into weekly orders and measure stock, service and cost
in a day-by-day replay of 2016-01-04 to 2016-05-22.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

from src.inventory.inputs import build_inputs, draw_lead_times
from src.inventory.policies import replenish_last_week
from src.inventory.simulate import simulate

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

inputs = build_inputs()
lead_times = draw_lead_times(inputs)
baseline = simulate(inputs, replenish_last_week, lead_times)
baseline.head()

## Baseline: reorder last week's sales

In [ ]:
scored = baseline[baseline["ds"] >= pd.Timestamp("2016-01-18")]  # skip the 14-day warm-up
by_class = scored.groupby("demand_class")[["demand", "sales", "end_stock"]].mean()
by_class["fill_rate"] = by_class["sales"] / by_class["demand"]
by_class["days_of_stock"] = by_class["end_stock"] / by_class["demand"]
by_class[["fill_rate", "days_of_stock"]].round(3)

In [ ]:
weekly = baseline.groupby(pd.Grouper(key="ds", freq="W-SUN"))[["demand", "lost"]].sum()

ax = (100 * weekly["lost"] / weekly["demand"]).plot(figsize=(10, 3.5), marker="o")
ax.set_title("Baseline: lost sales grow every week, because lost demand is never reordered")
ax.set_ylabel("Lost sales (% of demand)")
ax.set_xlabel("")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for ax, uid in zip(axes, ["FOODS_3_090_CA_1", "FOODS_3_004_CA_1"]):
    g = baseline[baseline["unique_id"] == uid]
    ax.plot(g["ds"], g["end_stock"], label="Stock at end of day")
    ax.bar(g["ds"], g["lost"], color="tab:red", label="Lost sales")
    ax.set_title(f"{uid}: average demand {g['demand'].mean():.1f} units per day")
    ax.set_ylabel("Units")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()